<div style='background:linear-gradient(135deg,#1B3A6B,#2D5AA0);padding:30px;border-radius:12px;color:white;font-family:Arial'>
<h1 style='margin:0;font-size:28px'>💬 Chapter 13 — Natural Language Processing Basics</h1>
<p style='margin:8px 0 0'>Book: Machine Learning — From Zero to Engineer | Est. Time: 75 mins | Level: Intermediate</p>
</div>

## 📋 Learning Objectives

By the end of this notebook, you will be able to:
- Tokenise and clean text (lowercasing, punctuation, stopwords)
- Create Bag-of-Words representations with CountVectorizer
- Build TF-IDF features to weight term importance
- Train a text classifier on real-world reviews
- Understand the limitations of traditional NLP and when to use deep learning

---

In [ ]:
# ─────────────────────────────────────────────────────────────
# Setup & Imports
# ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from collections import Counter

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model    import LogisticRegression
from sklearn.naive_bayes     import MultinomialNB
from sklearn.svm             import LinearSVC
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics         import classification_report, confusion_matrix
from sklearn.pipeline        import Pipeline

%matplotlib inline
np.random.seed(42)

# ── Synthetic Indian product review dataset ────────────────
reviews_positive = [
    'Excellent product, very happy with the quality',
    'Superb delivery from Flipkart, arrived in 2 days',
    'Best buy ever, great value for money in India',
    'Amazing quality, totally worth the price on Amazon India',
    'Love this product, my whole family is happy',
    'Fast delivery to Mumbai, product works perfectly',
    'Great quality at this price, highly recommend',
    'Happy with purchase, will order again on Flipkart',
    'Absolutely fantastic, looks exactly like photos',
    'Top quality item, arrived well packed to Bangalore',
    'Works great, very satisfied with this product',
    'Brilliant product for the price, highly recommend',
    'Very good quality, nice finish and fast shipping',
    'Perfect product received on time to Delhi',
    'Outstanding quality, exceeded my expectations',
] * 10

reviews_negative = [
    'Terrible quality, broke after one day',
    'Worst product ever, complete waste of money',
    'Very disappointed, not as described on Flipkart',
    'Do not buy this, poor quality material',
    'Received damaged item, no customer support',
    'Fake product, not original item at all',
    'Horrible experience, returned immediately',
    'Very bad quality, falls apart easily',
    'Cheated, sent wrong product from Amazon India',
    'Extremely poor quality, highly disappointed',
    'Pathetic product, waste of money do not buy',
    'Never buying again, very bad experience',
    'Worst purchase of my life, complete scam',
    'Defective item received, seller unresponsive',
    'Terrible, does not work as advertised at all',
] * 10

reviews = reviews_positive + reviews_negative
labels  = [1]*len(reviews_positive) + [0]*len(reviews_negative)

df_reviews = pd.DataFrame({'review': reviews, 'sentiment': labels})
df_reviews = df_reviews.sample(frac=1, random_state=42).reset_index(drop=True)
print(f'Dataset: {len(df_reviews)} reviews, positive: {labels.count(1)}, negative: {labels.count(0)}')
df_reviews.head()

## 📖 Section A — Text Preprocessing

Raw text must be cleaned before feature extraction:

```python
import re

def clean_text(text):
    text = text.lower()                         # lowercase
    text = re.sub(r'[^a-z\s]', '', text)        # remove punctuation/digits
    tokens = text.split()
    stopwords = {'the', 'is', 'in', 'on', 'a', 'an', 'and', 'to', 'for', 'of', 'with', 'this'}
    tokens = [t for t in tokens if t not in stopwords]
    return ' '.join(tokens)

df['clean'] = df['review'].apply(clean_text)
```

> 💡 **Key Rule:** For Hindi-English (Hinglish) reviews, simple rule-based cleaning works — but you may want to keep words like 'bahut', 'achha', 'bekar' as they carry strong sentiment. Always validate your cleaning on sample rows.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Demo: TF-IDF Features
# ─────────────────────────────────────────────────────────────
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    stopwords = {'the','is','in','on','a','an','and','to','for','of','with','this','my','very'}
    tokens = [t for t in text.split() if t not in stopwords]
    return ' '.join(tokens)

df_reviews['clean'] = df_reviews['review'].apply(clean_text)

tfidf = TfidfVectorizer(max_features=100, ngram_range=(1, 2))
X_tfidf = tfidf.fit_transform(df_reviews['clean'])
print(f'TF-IDF matrix shape: {X_tfidf.shape}')
print('Top 10 features:', tfidf.get_feature_names_out()[:10])

---
## 🟢 Exercise 13.1 — Bag-of-Words Sentiment Classifier *(Level 1 · Est. 10 min)*

> Build a Pipeline: CountVectorizer → MultinomialNB. Run 5-fold CV. Print accuracy and F1.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 13.1: Bag-of-Words + Naive Bayes
# ─────────────────────────────────────────────────────────────
X_text = df_reviews['clean']
y_sent = df_reviews['sentiment']

# YOUR CODE HERE — Pipeline with CountVectorizer + MultinomialNB

# YOUR CODE HERE — 5-fold CV accuracy and F1

print(f'BoW + Naive Bayes: Accuracy = {acc:.4f}, F1 = {f1:.4f}')

assert acc > 0.5, 'Accuracy should be above 0.5'
print('✅ Baseline classifier built!')

In [ ]:
# @title ✅ Solution — Exercise 13.1 (click to expand)

from sklearn.model_selection import StratifiedKFold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipe_nb = Pipeline([
    ('vec', CountVectorizer()),
    ('nb',  MultinomialNB())
])
acc = cross_val_score(pipe_nb, X_text, y_sent, cv=cv, scoring='accuracy').mean()
f1  = cross_val_score(pipe_nb, X_text, y_sent, cv=cv, scoring='f1').mean()

print(f'BoW + Naive Bayes: Accuracy = {acc:.4f}, F1 = {f1:.4f}')
print('✅ Naive Bayes is fast and often the best baseline for text classification.')

---
## 🟡 Exercise 13.2 — TF-IDF with Multiple Classifiers *(Level 2 · Est. 20 min)*

> Compare TF-IDF + (Logistic Regression, LinearSVC, MultinomialNB) using 5-fold CV. Include unigrams and bigrams. Report accuracy and F1 in a table.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 13.2: TF-IDF + Multiple Classifiers
# ─────────────────────────────────────────────────────────────
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'LinearSVC':           LinearSVC(max_iter=2000),
    'Naive Bayes':         MultinomialNB(),
}

results = []
# YOUR CODE HERE — for each classifier, build TF-IDF pipeline, run 5-fold CV

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
print('✅ Text classifier comparison done!')

In [ ]:
# @title ✅ Solution — Exercise 13.2 (click to expand)

classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'LinearSVC':           LinearSVC(max_iter=2000),
    'Naive Bayes':         MultinomialNB(),
}

results = []
for name, clf in classifiers.items():
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=5000)),
        ('clf',   clf)
    ])
    acc = cross_val_score(pipe, X_text, y_sent, cv=cv, scoring='accuracy').mean()
    f1  = cross_val_score(pipe, X_text, y_sent, cv=cv, scoring='f1').mean()
    results.append({'Classifier': name, 'Accuracy': round(acc, 4), 'F1': round(f1, 4)})

results_df = pd.DataFrame(results).sort_values('F1', ascending=False)
print(results_df.to_string(index=False))
print('\n✅ LinearSVC often best for text: handles high-dimensional sparse features well.')

---
## 🔴 Exercise 13.3 — Error Analysis and Top Keywords *(Level 3 · Est. 20 min)*

> Fit the best pipeline on train set, predict on test. Show confusion matrix. Print top 10 positive and negative keywords by TF-IDF coefficient. Show 5 misclassified reviews.

In [ ]:
# @title ✅ Solution — Exercise 13.3 (click to expand)

X_train, X_test, y_train, y_test = train_test_split(X_text, y_sent, test_size=0.2, random_state=42, stratify=y_sent)

best_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=5000)),
    ('clf',   LogisticRegression(max_iter=1000))
])
best_pipe.fit(X_train, y_train)
y_pred = best_pipe.predict(X_test)

print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative','Positive'], yticklabels=['Negative','Positive'])
plt.title('Confusion Matrix — Sentiment Classifier')
plt.tight_layout()
plt.show()

# Top keywords
import seaborn as sns
feat_names = best_pipe.named_steps['tfidf'].get_feature_names_out()
coefs = best_pipe.named_steps['clf'].coef_[0]
top_pos = pd.Series(coefs, index=feat_names).nlargest(10)
top_neg = pd.Series(coefs, index=feat_names).nsmallest(10)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
top_pos.plot(kind='barh', ax=axes[0], color='green'); axes[0].set_title('Top Positive Keywords')
top_neg.abs().plot(kind='barh', ax=axes[1], color='red'); axes[1].set_title('Top Negative Keywords')
plt.tight_layout()
plt.show()

# Misclassified reviews
test_df = pd.DataFrame({'review': X_test.values, 'true': y_test.values, 'pred': y_pred})
misclassified = test_df[test_df['true'] != test_df['pred']]
print(f'\nMisclassified: {len(misclassified)}')
print(misclassified.head(5)[['review', 'true', 'pred']].to_string())

---
## 🎤 Interview Q&A

<details>
<summary><strong>Q1: What is the difference between Bag-of-Words and TF-IDF?</strong></summary>

**Answer:** Bag-of-Words counts raw word frequencies in each document. TF-IDF weights each word by TF (how often it appears in THIS document) × IDF (inverse of how many documents contain it). IDF downweights common words like 'the', 'is', 'product' that appear in almost every document and carry little discriminative power. TF-IDF highlights words that are frequent in a document but rare across all documents — these are usually the most meaningful keywords.
</details>

<details>
<summary><strong>Q2: Why is Naive Bayes effective for text classification?</strong></summary>

**Answer:** Naive Bayes assumes feature independence — each word contributes independently to the class probability. For text, word co-occurrence dependencies exist but this simplification still works well in practice. NB has several advantages for text: (1) Works well with sparse high-dimensional features, (2) Very fast to train O(features × classes), (3) Needs less training data than discriminative models, (4) Works well for short texts. MultinomialNB is specifically designed for word counts.
</details>

<details>
<summary><strong>Q3: What are ngrams and why use bigrams?</strong></summary>

**Answer:** An n-gram is a contiguous sequence of n words. Unigrams (n=1): 'not', 'good'. Bigrams (n=2): 'not good', 'highly recommend'. Bigrams capture negation and phrase context that unigrams miss. 'not good' is very different from 'not' + 'good'. Trigrams (n=3) are less common — they add more context but increase dimensionality drastically. ngram_range=(1,2) in TF-IDF includes both unigrams and bigrams.
</details>

---

<div style='background:#EDF7F0;border-left:6px solid #2E7D50;padding:20px;border-radius:8px;margin-top:20px'>
<h3 style='color:#2E7D50'>✅ Chapter 13 Complete!</h3>
<ul>
<li>Text preprocessing: cleaning, tokenisation, stopwords</li>
<li>Bag-of-Words and TF-IDF feature extraction</li>
<li>Sentiment classification with Naive Bayes, LR, LinearSVC</li>
<li>Error analysis and keyword importance</li>
</ul>
<p><strong>Next:</strong> Chapter 14 — Introduction to Neural Networks</p>
</div>